# Imports

In [73]:
import boto3
import os
from sagemaker.core import image_uris
from sagemaker.core.parameter import ContinuousParameter, IntegerParameter
from sagemaker.core.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.core.shapes.shapes import ProcessingS3Input, ProcessingS3Output
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.workflow.functions import Join
from sagemaker.core.workflow.execution_variables import ExecutionVariables
from sagemaker.core.model_metrics import ModelMetrics, MetricsSource
from sagemaker.mlops import ModelBuilder
from sagemaker.mlops.workflow.model_step import ModelStep
from sagemaker.mlops.workflow.steps import ProcessingStep, TuningStep
from sagemaker.train import ModelTrainer
from sagemaker.train.tuner import HyperparameterTuner
from sagemaker.core.training.configs import Compute, InputData, OutputDataConfig, SourceCode
from botocore.exceptions import WaiterError

# Initialization

In [74]:
def list_all_sagemaker_domains(region_name='eu-west-1'):
    """
    List all SageMaker domains in the current AWS account and region.

    Returns:
        list: A list of dictionaries containing domain details.
    """
    client = boto3.client('sagemaker',region_name=region_name)
    response = client.list_domains()
    return response.get('Domains', [])

def get_sagemaker_domain_arn(domain_name, region_name='eu-west-1'):
    """
    Get the ARN of a specific SageMaker domain by its name.

    Args:
        domain_name (str): The name of the SageMaker domain.
        region_name (str): The AWS region where the domain is located.

    Returns:
        str: The ARN of the specified SageMaker domain, or None if not found.
    """
    domains = list_all_sagemaker_domains(region_name)
    for domain in domains:
        if domain['DomainName'] == domain_name:
            return domain['DomainArn']
    return None

In [75]:
list_of_domains = list_all_sagemaker_domains()
print("List of SageMaker Domains:", list_of_domains)
print("ARN of the first domain:", get_sagemaker_domain_arn(list_of_domains[0]['DomainName']) if list_of_domains else "No domains found")

List of SageMaker Domains: [{'DomainArn': 'arn:aws:sagemaker:eu-west-1:746760141372:domain/d-shjr7ddqgndd', 'DomainId': 'd-shjr7ddqgndd', 'DomainName': 'QuickSetupDomain-20261009T224547', 'Status': 'InService', 'CreationTime': datetime.datetime(2026, 10, 9, 19, 45, 45, 943000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 10, 9, 19, 45, 55, 999000, tzinfo=tzlocal()), 'Url': 'https://d-shjr7ddqgndd.studio.eu-west-1.sagemaker.aws'}, {'DomainArn': 'arn:aws:sagemaker:eu-west-1:746760141372:domain/d-mvykom11e1ij', 'DomainId': 'd-mvykom11e1ij', 'DomainName': 'QuickSetupDomain-20260924T193773', 'Status': 'InService', 'CreationTime': datetime.datetime(2026, 9, 24, 16, 37, 48, 627000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 9, 24, 16, 38, 5, 153000, tzinfo=tzlocal()), 'Url': 'https://d-mvykom11e1ij.studio.eu-west-1.sagemaker.aws'}]
ARN of the first domain: arn:aws:sagemaker:eu-west-1:746760141372:domain/d-shjr7ddqgndd


In [76]:
pipeline_session = PipelineSession()
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
account_id = boto3.client('sts').get_caller_identity().get('Account')
print(f"Region: {region}, Account ID: {account_id}")
role = f"arn:aws:iam::746760141372:role/service-role/AmazonSageMaker-ExecutionRole-20260924T193773"
domain_name = os.getenv('DOMAIN_NAME') 
print(f"Domain Name from environment variable: {domain_name}")
domain_arn = get_sagemaker_domain_arn(domain_name) if domain_name else None
print(f"Domain ARN: {domain_arn if domain_arn else 'Domain not found'}")
print(f"role: {role}")

Region: eu-west-1, Account ID: 746760141372
Domain Name from environment variable: QuickSetupDomain-20260924T193773
Domain ARN: arn:aws:sagemaker:eu-west-1:746760141372:domain/d-mvykom11e1ij
role: arn:aws:iam::746760141372:role/service-role/AmazonSageMaker-ExecutionRole-20260924T193773


# Pipeline Parameters

In [77]:
# S3 data locations and registry destination
raw_data_s3_uri = ParameterString(
    name="RawDataS3Uri",
    default_value="s3://driftwatch-bucket/data/raw/telco_churn.csv",
)
processed_data_s3_uri = ParameterString(
    name="ProcessedDataS3Uri",
    default_value="s3://driftwatch-bucket/data/processed",
)
processed_model_s3_uri = ParameterString(
    name="ProcessedModelS3Uri",
    default_value="s3://driftwatch-bucket/models",
)

model_package_group_name = ParameterString(
    name="ModelPackageGroupName",
    default_value="DriftWatchChurn",
)


# Processor

In [78]:
script_processor = ScriptProcessor(
    command=["python3"],
    role=role,
    image_uri=image_uris.retrieve(
        framework="sklearn",
        region=region,
        version="1.2-1",
        instance_type="ml.t3.medium",
    ),
    instance_count=1,
    instance_type="ml.t3.medium",
    base_job_name="driftwatch-processing-job",
    sagemaker_session=pipeline_session,
)

[10/17/26 21:51:49] INFO     Defaulting to only available Python version: py3                     ]8;id=3936787;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3936788;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

# Step 1 (Preprocessing)

In [79]:
step_preprocessing = ProcessingStep(
    name="PreprocessingStep",
    step_args=script_processor.run(
        code="steps/preprocess/preprocess.py",
        arguments=[
            "--input_s3_uri", raw_data_s3_uri,
            "--region", region,
            "--output_dir", "/opt/ml/processing/output",
        ],
        outputs=[
            ProcessingOutput(
                output_name="preprocessing",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(on="/", values=[processed_data_s3_uri, "preprocessing"]),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/preprocessing",
                ),
            ),
            ProcessingOutput(
                output_name="train",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(
                        on="/", values=[processed_data_s3_uri, "train"]
                    ),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/train",
                ),
            ),
            ProcessingOutput(
                output_name="validation",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(
                        on="/", values=[processed_data_s3_uri, "validate"]
                    ),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/validate",
                ),
            ),
            ProcessingOutput(
                output_name="test",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(
                        on="/", values=[processed_data_s3_uri, "test"]
                    ),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/test",
                ),
            ),
            ProcessingOutput(
                output_name="baseline",
                s3_output=ProcessingS3Output(
                    s3_uri=Join(
                        on="/", values=[processed_data_s3_uri, "baseline"]
                    ),
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/baseline",
                ),
            ),
        ],
    ),
)

# Step 2 (Bayesian Tuning)

In [80]:
# Use a framework container so AMT accepts the custom best-iteration metric.
training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1",
    image_scope="training",
    instance_type="ml.m5.large",
)

# Processing jobs do not install the training source requirements.
evaluation_image = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    instance_type="ml.m5.large",
)

model_trainer = ModelTrainer(
    role=role,
    sagemaker_session=pipeline_session,
    base_job_name="driftwatch-training-job",
    source_code=SourceCode(
        source_dir="steps/train",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    training_image=training_image,
    compute=Compute(
        instance_type="ml.m5.large",
        instance_count=1,
    ),
    output_data_config=OutputDataConfig(
        s3_output_path=Join(on="/", values=[processed_model_s3_uri, "artifacts"]),
    ),
    hyperparameters={
        "train-path": "/opt/ml/input/data/train/train.csv",
        "validation-path": "/opt/ml/input/data/validation/validation.csv",
        "model-dir": "/opt/ml/model",
        "preprocessing-dir": "/opt/ml/input/data/preprocessing",
    },
)

training_inputs = [
    InputData(
        channel_name="preprocessing",
        data_source=step_preprocessing.properties.ProcessingOutputConfig.Outputs[
            "preprocessing"
        ].S3Output.S3Uri,
    ),
    InputData(
        channel_name="train",
        data_source=step_preprocessing.properties.ProcessingOutputConfig.Outputs[
            "train"
        ].S3Output.S3Uri,
    ),
    InputData(
        channel_name="validation",
        data_source=step_preprocessing.properties.ProcessingOutputConfig.Outputs[
            "validation"
        ].S3Output.S3Uri,
    ),
]

tuner = HyperparameterTuner(
    model_trainer=model_trainer,
    strategy="Bayesian",
    objective_metric_name="validation:best_aucpr",
    objective_type="Maximize",
    hyperparameter_ranges={
        "max_depth": IntegerParameter(3, 8, scaling_type="Linear"),
        "eta": ContinuousParameter(0.05, 0.20, scaling_type="Logarithmic"),
        "min_child_weight": ContinuousParameter(1, 10, scaling_type="Logarithmic"),
    },
    metric_definitions=[{
        "Name": "validation:best_aucpr",
        "Regex": r"HPO_METRIC best_aucpr=([0-9]+\.[0-9]+)",
    }],
    max_jobs=10,
    max_parallel_jobs=1,
    random_seed=42,
)

step_tuning = TuningStep(
    name="BayesianTuningStep",
    step_args=tuner.tune(inputs=training_inputs),
)

best_model_s3_uri = Join(on="/", values=[
    processed_model_s3_uri,
    "artifacts",
    step_tuning.properties.TrainingJobSummaries[0].TrainingJobName,
    "output/model.tar.gz",
])

                    INFO     Defaulting to only available Python version: py3                     ]8;id=3936793;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3936794;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Ignoring unnecessary instance type: ml.m5.large.                     ]8;id=3936799;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3936800;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=3936805;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3936806;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=3936811;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3936812;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=3936817;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=3936818;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             141502667606.dkr.ecr.eu-west-1.amazonaws.com/sagemaker-scikit-lea                     
                             rn:1.2-1-cpu-py3                                                                      

# Step 3 (Evaluation)

In [81]:
# Step 3 (Evaluation)
evaluation_processor = ScriptProcessor(
    command=["python3"],
    role=role,
    image_uri=evaluation_image,
    instance_count=1,
    instance_type="ml.t3.medium",
    base_job_name="driftwatch-evaluation-job",
    sagemaker_session=pipeline_session,
)

evaluation_output_path = Join(
    on="/", values=[
        processed_model_s3_uri, "evaluation", ExecutionVariables.PIPELINE_EXECUTION_ID
    ]
)

step_evaluation = ProcessingStep(
    name="EvaluationStep",
    step_args=evaluation_processor.run(
        code="steps/evaluate/evaluate.py",
        inputs=[
            ProcessingInput(
                input_name="test",
                s3_input=ProcessingS3Input(
                    s3_uri=step_preprocessing.properties.ProcessingOutputConfig.Outputs[
                        "test"
                    ].S3Output.S3Uri,
                    s3_data_type="S3Prefix",
                    s3_input_mode="File",
                    local_path="/opt/ml/processing/input/test",
                ),
            ),
            ProcessingInput(
                input_name="model",
                s3_input=ProcessingS3Input(
                    s3_uri=best_model_s3_uri,
                    s3_data_type="S3Prefix",
                    s3_input_mode="File",
                    local_path="/opt/ml/processing/input/model",
                ),
            ),
        ],
        arguments=[
            "--test-path", "/opt/ml/processing/input/test",
            "--model-dir", "/opt/ml/processing/input/model",
            "--output-dir", "/opt/ml/processing/output/evaluation",
        ],
        outputs=[
            ProcessingOutput(
                output_name="evaluation",
                s3_output=ProcessingS3Output(
                    s3_uri=evaluation_output_path,
                    s3_upload_mode="EndOfJob",
                    local_path="/opt/ml/processing/output/evaluation",
                ),
            ),
        ],
    ),
)

# Step 4 (Model Registry)

In [82]:
# Register only after evaluation succeeds; no metric threshold gate.
inference_image = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    image_scope="inference",
    instance_type="ml.m5.large",
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(on="/", values=[
            step_evaluation.properties.ProcessingOutputConfig.Outputs[
                "evaluation"
            ].S3Output.S3Uri,
            "evaluation.json",
        ]),
        content_type="application/json",
    ),
)

registry_model = ModelBuilder(
    role_arn=role,
    sagemaker_session=pipeline_session,
    image_uri=inference_image,
    s3_model_data_url=best_model_s3_uri,
    # A standalone entry script is packaged into its own source archive by the SDK.
    # Preprocessing code/state stay in the winning model archive. No repack is needed.
    source_code=SourceCode(entry_script="steps/inference/inference.py"),
)

step_registration = ModelStep(
    name="RegisterWinningModel",
    step_args=registry_model.register(
        model_package_group_name=model_package_group_name,
        approval_status="PendingManualApproval",
        content_types=["application/json", "text/csv"],
        response_types=["application/json"],
        model_metrics=model_metrics,
    ),
    depends_on=[step_evaluation],
)


[10/17/26 21:51:50] INFO     Ignoring unnecessary instance type: ml.m5.large.                     ]8;id=3936823;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3936824;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

                    DEBUG    Auto-detecting optimal instance type for model...           ]8;id=3936829;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=3936830;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#340\340]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=3936835;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=3936836;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#374\374]8;;\

# Pipeline Creation

In [83]:
pipeline_name = "DriftWatchPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        raw_data_s3_uri,
        processed_data_s3_uri,
        processed_model_s3_uri,
        model_package_group_name,
    ],
    steps=[
        step_preprocessing, step_tuning, step_evaluation,
        *step_registration.steps, step_registration,
    ],
    sagemaker_session=pipeline_session,
)

In [84]:
# Provision the default group before submitting the pipeline. Runtime overrides
# of ModelPackageGroupName must refer to an existing group in this region.
registry_client = pipeline_session.sagemaker_client
default_group_name = model_package_group_name.default_value
group_pages = registry_client.get_paginator("list_model_package_groups").paginate(
    NameContains=default_group_name,
)
if not any(
    group["ModelPackageGroupName"] == default_group_name
    for page in group_pages
    for group in page["ModelPackageGroupSummaryList"]
):
    registry_client.create_model_package_group(
        ModelPackageGroupName=default_group_name,
        ModelPackageGroupDescription="DriftWatch customer churn models",
    )

pipeline.upsert(
    role_arn=role,
    tags = [
        {"Key": "sagemaker:domain-arn", "Value": domain_arn},
    ]
)

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=3936841;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3936842;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[10/17/26 21:51:51] WARNING  Popping out 'HyperParameterTuningJobName' from the pipeline           ]8;id=3936847;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3936848;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             definition by default since it will be overridden at pipeline                         
                             execution time. Please utilize the PipelineDefinitionConfig to                        
                             persist this field in the pipeline definition if desired.                             

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=3936853;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3936854;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=3936859;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=3936860;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=3936865;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3936866;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=3936871;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3936872;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[10/17/26 21:51:52] WARNING  Popping out 'HyperParameterTuningJobName' from the pipeline           ]8;id=3936877;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3936878;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             definition by default since it will be overridden at pipeline                         
                             execution time. Please utilize the PipelineDefinitionConfig to                        
                             persist this field in the pipeline definition if desired.                             

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=3936883;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3936884;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=3936889;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=3936890;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

{'PipelineArn': 'arn:aws:sagemaker:eu-west-1:746760141372:pipeline/DriftWatchPipeline',
 'PipelineVersionId': 23,
 'ResponseMetadata': {'RequestId': '89ddd487-114a-4249-b81f-1ba2c2113ebe',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '89ddd487-114a-4249-b81f-1ba2c2113ebe',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '109',
   'date': 'Sat, 17 Oct 2026 21:51:52 GMT'},
  'RetryAttempts': 0}}

In [85]:
execution = pipeline.start()

try:
    execution.wait()
except WaiterError:
    print("Pipeline execution failed. Fetching failure details...")

execution_description = execution.describe()
execution_status = execution_description["PipelineExecutionStatus"]

print(f"Pipeline execution status: {execution_status}")

if execution_status != "Succeeded":
    print(
        "Pipeline failure reason:",
        execution_description.get("FailureReason", "Not provided")
    )

    # Get failure details for each step
    steps = execution.list_steps()

    for step in steps:
        print(f"\nStep: {step['StepName']}")
        print(f"Status: {step['StepStatus']}")

        if step["StepStatus"] == "Failed":
            print(
                "Failure reason:",
                step.get("FailureReason", "Not provided")
            )

    raise RuntimeError(
        f"DriftWatchPipeline execution failed with status: {execution_status}"
    )

print("Pipeline executed successfully!")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│    1 execution = pipeline.start()                                                                │
│    2                                                                                             │
│    3 try:                                                                                        │
│ ❱  4 │   execution.wait()                                                                        │
│    5 except WaiterError:                                                                         │
│    6 │   print("Pipeline execution failed. Fetching failure details...")                         │
│    7                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/pipeline.py:1075 in wait        │
│                                                                                                  │
│   1072 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│   1073 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│   1074 │   │   )                                                                                 │
│ ❱ 1075 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│   1076 │                                                                                         │
│   1077 │   def result(self, step_name: str):                                                     │
│   1078 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:58 in wait                            │
│                                                                                                  │
│    55 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    56 │   # method.                                                                              │
│    57 │   def wait(self, **kwargs):                                                              │
│ ❱  58 │   │   Waiter.wait(self, **kwargs)                                                        │
│    59 │                                                                                          │
│    60 │   wait.__doc__ = WaiterDocstring(                                                        │
│    61 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                      